# Controlli aggiuntivi del modello nominale

Due controlli promessi nella proposta e non presenti in `Bayesian_Nominal.ipynb` (che non viene rieseguito):

1. **Posterior predictive check** della distribuzione di massa: per $\theta$ estratti dalla posterior si genera un dataset replicato e si confronta una discrepanza $T(y,\theta)$ calcolata sui dati e sulle repliche; $p_B=P(T_{\rm rep}\ge T_{\rm obs})$. Discrepanze: $\chi^2$ binnato (bin da 10 MeV, entrambe le cariche), asimmetria di conteggio nell'intera finestra, nella finestra di segnale e nelle sideband.
2. **Trace plot**: catena del modello nominale con walker inizializzati uniformemente nella scatola della prior (25000 passi), salvata ogni 10 passi.

Legge `results/nominal/nominal_posterior_thinned.npy` e `nominal_summary.json`; scrive `results/nominal/ppc.json` e `results/nominal/trace_dispersed.npz`. Ogni blocco riprende se il file esiste.


In [1]:
import os, sys, json
import numpy as np
sys.path.insert(0, ".")
import bayesian_nominal as bn
import emcee

DATA_DIR = "../../opendata-project/Data"
RESULTS_DIR = os.path.join(os.getcwd(), "results", "nominal")
cfg = bn.NOMINAL
data = bn.select(bn.load_candidates(DATA_DIR), cfg)
m_obs, q_obs = data.m.to_numpy(), data.q.to_numpy()
summary = json.load(open(os.path.join(RESULTS_DIR, "nominal_summary.json")))
names = summary["nominal"]["names"]
thinned = np.load(os.path.join(RESULTS_DIR, "nominal_posterior_thinned.npy"))
MU, SIG = summary["nominal"]["mu"]["median"], summary["nominal"]["sigma"]["median"]
print(len(m_obs), thinned.shape)


19731 (20000, 12)


## Posterior predictive check

In [2]:
N_REP = 1000
def count_asym(q):
    return float((np.sum(q < 0) - np.sum(q > 0)) / len(q)) if len(q) else float("nan")
def discrepancies(m, q, theta):
    sw = np.abs(m - MU) < 2 * SIG
    sb = np.abs(m - MU) > 4 * SIG
    g = bn.binned_gof(bn.Posterior(m, q, cfg), theta)
    return {"chi2": g["chi2"], "chi2_pos": g["per_charge"]["pos"], "chi2_neg": g["per_charge"]["neg"],
            "A_window": count_asym(q), "A_signal_window": count_asym(q[sw]), "A_sidebands": count_asym(q[sb])}

path = os.path.join(RESULTS_DIR, "ppc.json")
if os.path.exists(path):
    ppc = json.load(open(path)); print("[resume]", path)
else:
    rng = np.random.default_rng(2026)
    idx = rng.choice(len(thinned), N_REP, replace=False)
    rows = []
    for k, i in enumerate(idx):
        th = thinned[i]
        m_rep, q_rep = bn.generate_toy(dict(zip(names, th)), cfg, rng)
        rows.append({"obs": discrepancies(m_obs, q_obs, th), "rep": discrepancies(m_rep, q_rep, th)})
        if (k + 1) % 200 == 0:
            print(k + 1)
    keys = list(rows[0]["obs"])
    ppc = {"n_rep": N_REP, "rows": rows, "p_B": {}}
    for key in keys:
        o = np.array([r["obs"][key] for r in rows]); r_ = np.array([r["rep"][key] for r in rows])
        if key.startswith("chi2"):
            ppc["p_B"][key] = float(np.mean(r_ >= o))           # one-sided: misfit = large chi2
        else:
            ppc["p_B"][key] = float(2 * min(np.mean(r_ >= o), np.mean(r_ <= o)))   # two-sided
    json.dump(ppc, open(path, "w"), indent=1); print("[saved]", path)
for key, p in ppc["p_B"].items():
    o = np.array([r["obs"][key] for r in ppc["rows"]]); r_ = np.array([r["rep"][key] for r in ppc["rows"]])
    print(f"{key:16s} T_obs = {np.median(o):9.4f}   T_rep = {r_.mean():9.4f} +- {r_.std():.4f}   p_B = {p:.3f}")


200


400


600


800


1000
[saved] /Users/gmoretti/Work/wt_bayes/LHCb/Code_general/Bayesian/results/nominal/ppc.json
chi2             T_obs =  174.6950   T_rep =  103.8398 +- 14.7864   p_B = 0.000
chi2_pos         T_obs =  100.5458   T_rep =   51.8853 +- 10.3877   p_B = 0.000
chi2_neg         T_obs =   73.5755   T_rep =   51.9545 +- 10.4754   p_B = 0.030
A_window         T_obs =   -0.0432   T_rep =   -0.0432 +- 0.0098   p_B = 0.996
A_signal_window  T_obs =   -0.0461   T_rep =   -0.0465 +- 0.0110   p_B = 0.992
A_sidebands      T_obs =   -0.0106   T_rep =   -0.0182 +- 0.0246   p_B = 0.734


## Trace plot (partenza dispersa)

In [3]:
path_tr = os.path.join(RESULTS_DIR, "trace_dispersed.npz")
if os.path.exists(path_tr):
    print("[resume]", path_tr)
else:
    post = bn.Posterior(m_obs, q_obs, cfg)
    lo_, hi_ = bn._prior_box(cfg, post.n, names)
    rng = np.random.default_rng(7)
    pos = lo_ + (hi_ - lo_) * rng.uniform(0.02, 0.98, (32, len(names)))
    for w in range(32):                       # same rule as bn.fit: every walker starts inside the prior
        while not np.isfinite(post(pos[w])):
            pos[w] = lo_ + (hi_ - lo_) * rng.uniform(0.02, 0.98, len(names))
    np.random.seed(7)
    sam = emcee.EnsembleSampler(32, len(names), post)
    sam.run_mcmc(pos, 25000, progress=False)
    ch = sam.get_chain()[::10].astype(np.float32)          # (2500, 32, 12)
    np.savez_compressed(path_tr, chain=ch, names=np.array(names), thin=10, lnp=sam.get_log_prob()[::10].astype(np.float32))
    print("[saved]", path_tr, ch.shape)
tr = np.load(path_tr)
ch = tr["chain"]; half = ch.shape[0] // 2
iA = list(tr["names"]).index("A")
print("A, second half of the chain: median", float(np.median(ch[half:, :, iA])), " sd", float(np.std(ch[half:, :, iA])))


[saved] /Users/gmoretti/Work/wt_bayes/LHCb/Code_general/Bayesian/results/nominal/trace_dispersed.npz (2500, 32, 12)
A, second half of the chain: median -0.0440354198217392  sd 0.00856824778020382
